# KV Cache & PagedAttention

> **Status:** Content notebook — theory complete, hands-on exercises require a GPU runtime.

## Learning Objectives

By the end of this notebook, you will be able to:

- [ ] Explain the Key-Value (KV) cache and why it exists in autoregressive decoding
- [ ] Describe the memory layout of KV cache tensors and how they scale with batch size and sequence length
- [ ] Understand the memory fragmentation problem that PagedAttention solves
- [ ] Explain how vLLM's PagedAttention manages non-contiguous KV blocks
- [ ] Estimate KV cache memory requirements for a given model and request profile
- [ ] Identify when KV cache is the bottleneck vs compute

---

## Prerequisites

- [01_START_HERE.ipynb](../01_START_HERE/01_START_HERE.ipynb)
- Basic understanding of the transformer architecture (Phase 6)
- Familiarity with GPU memory concepts

---

## 1. Why Does a KV Cache Exist?

During autoregressive decoding, a transformer generates one token at a time.
At each step, the attention mechanism computes queries, keys, and values for
**all** positions in the sequence so far. Without caching, the K and V tensors
for earlier tokens would be recomputed at every step.

The **KV cache** stores the K and V projections for all previously generated tokens,
so each decoding step only computes K/V for the **new** token and appends it to the cache.

```
Memory per token (per layer) = 2 × (num_heads × head_dim) × dtype_bytes
```

For Llama-3 8B (32 layers, 32 heads, head_dim=128, float16):
- Per token, per layer = 2 × (32 × 128) × 2 bytes = **16 KB**
- For 4096-token sequence, full cache = 16 KB × 32 layers = **2 GB**

---

## 2. The Memory Fragmentation Problem

Traditional KV cache implementations pre-allocate a **contiguous block** of GPU memory
per request, sized for the maximum possible sequence length. This causes:

1. **Internal fragmentation** — short requests waste reserved memory
2. **No memory sharing** — parallel decoding cannot share prefix cache blocks
3. **Poor batching** — unpredictable actual lengths make batching inefficient

---

## 3. PagedAttention

PagedAttention (introduced in vLLM, OSDI 2023) manages KV cache using **fixed-size blocks**
(pages), similar to virtual memory paging in operating systems.

Key design choices:
- Each block holds a fixed number of tokens (block_size = 16 or 32 typically)
- Blocks are allocated on-demand, not pre-allocated
- Non-contiguous physical blocks are mapped through a **block table**
- Prefix blocks can be **shared** across requests with the same prefix

```
Block Table for Request A:
  logical block 0 → physical block 7
  logical block 1 → physical block 2
  logical block 2 → physical block 15 (still being filled)
```

---

## 4. Memory Layout Calculation

The GPU memory budget for KV cache is whatever remains after loading model weights.

```
kv_cache_bytes = (gpu_memory_total - model_weight_bytes) × gpu_memory_utilization
num_blocks = kv_cache_bytes // (block_size × 2 × num_layers × num_heads × head_dim × dtype_bytes)
```

vLLM computes this automatically at startup and reports:
```
# vLLM startup log example
INFO: # GPU blocks: 2048, # CPU blocks: 512
```

---

## 5. Measuring KV Cache Pressure

```python
# Inspect vLLM KV cache stats (requires vLLM installed with GPU)
from vllm import LLM, SamplingParams

llm = LLM(model="meta-llama/Meta-Llama-3-8B-Instruct", gpu_memory_utilization=0.90)

# After inference, check cache usage via metrics
# llm.llm_engine.scheduler.block_manager.get_num_free_gpu_blocks()
```

---

## 6. Key Metrics

| Metric | What it measures | Target |
|--------|------------------|--------|
| KV cache hit rate | Fraction of requests reusing cached prefixes | > 50% for shared-prefix workloads |
| GPU block utilisation | Fraction of allocated blocks in use | > 80% |
| Cache eviction rate | How often blocks are swapped to CPU | Should be near 0 in steady state |
| TTFT (Time to First Token) | Latency from request to first output token | Depends on SLO; typically < 500ms |

---

## Exercises

1. Estimate the KV cache memory requirement for a Llama-3 70B model at 8K context length.
2. Explain what happens to memory usage as you increase `gpu_memory_utilization` from 0.7 to 0.95.
3. Describe a workload where prefix caching provides the most benefit.

---

## References

- [PagedAttention paper (Kwon et al., OSDI 2023)](https://arxiv.org/abs/2309.06180)
- [vLLM documentation — Memory Management](https://docs.vllm.ai/en/latest/design/arch_overview.html)
- [Continuous Batching blog (Anyscale)](https://www.anyscale.com/blog/continuous-batching-llm-inference)

## What Comes Next

- [04_quantization_deep_dive.ipynb](../04_quantization_deep_dive/04_quantization_deep_dive.ipynb) — Reduce model weight memory with AWQ and GPTQ
- [05_speculative_decoding.ipynb](../05_speculative_decoding/05_speculative_decoding.ipynb) — Speed up decode latency
